# 04 — Training Walk-Forward CV

Train TCN+GRU folds with train-only scaling, HMM diagnostics, and artifact export.

In [ ]:
import torch, sys
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime → T4"
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_BASE  = '/content/drive/MyDrive/yeniBot'
DATA_DIR    = f'{DRIVE_BASE}/data'
CHECKPT_DIR = f'{DRIVE_BASE}/checkpoints'
os.makedirs(f'{DATA_DIR}/processed', exist_ok=True)
os.makedirs(CHECKPT_DIR, exist_ok=True)

In [ ]:
import os, sys
REPO_URL = 'https://github.com/umutergul74/yeniBot.git'
REPO_DIR = '/content/yenibot_repo'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
sys.path.insert(0, REPO_DIR)
print('After git pull, use Runtime → Restart session before trusting changed imports.')

In [ ]:
!pip install -q -r {REPO_DIR}/requirements.txt

In [ ]:
import yaml
with open(f'{REPO_DIR}/config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['paths']['data_dir'] = DATA_DIR
cfg['paths']['checkpoint_dir'] = CHECKPT_DIR

In [ ]:
import pandas as pd
from google.colab import runtime
from yenibot.experiments import experiment_settings, prepare_training_holdout_split, run_experiment_matrix

labeled_full = pd.read_parquet(f'{DATA_DIR}/processed/labeled_1h.parquet')
holdout_path = f'{DATA_DIR}/processed/holdout_1h.parquet'
labeled, holdout, holdout_meta = prepare_training_holdout_split(
    labeled_full,
    cfg,
    holdout_path=holdout_path,
)
min_selection_rows = (
    cfg['walk_forward']['train_bars']
    + cfg['walk_forward']['val_bars']
    + cfg['walk_forward']['test_bars']
    + cfg['walk_forward']['purge_bars']
    + cfg['walk_forward']['embargo_bars']
)
if len(labeled) <= min_selection_rows:
    raise ValueError(
        f'Not enough selection rows for one full CV fold after holdout split: {len(labeled)} rows'
    )
cfg.setdefault('experiments', {})['holdout'] = holdout_meta

print('Full labeled rows:', len(labeled_full))
print('Selection/training rows:', len(labeled))
print('Holdout rows:', len(holdout))
print('Selection data end:', holdout_meta['selection_data_end'])
print('Holdout data start:', holdout_meta['holdout_data_start'])
print('Holdout split mode:', holdout_meta.get('split_mode'))
print('Unused rows after anchor:', holdout_meta.get('unused_rows_after_anchor'))
print('Future OOS ready:', holdout_meta.get('future_oos_ready'))
print('Future OOS next action:', holdout_meta.get('next_action'))
print('Holdout saved:', holdout_path)
settings = experiment_settings(cfg)
print('Experiment mode:', settings.get('mode', 'staged'))
print('Active feature profile:', cfg.get('features', {}).get('active_profile'))
print('Control profile:', settings.get('control_profile'))
print('Candidate profiles:', settings.get('candidate_profiles', []))
print('Skipped historical profiles:', settings.get('skipped_profiles', []))
print('Full CV mode:', settings.get('full_cv_profiles', 'auto'))
print('Always-full profiles:', settings.get('always_full_profiles', []))
print('Max auto full candidates:', settings.get('max_auto_full_candidates'))
print('Seed audit:', settings.get('seed_audit', {}))
print('Seed:', cfg['project']['random_seed'], 'deterministic:', cfg['project'].get('deterministic', False))

try:
    result = run_experiment_matrix(
        labeled,
        cfg,
        checkpoint_dir=CHECKPT_DIR,
    )
    print('Experiment run:', result['run_id'])
    print('Experiment dir:', result['run_dir'])
    print('Run id source:', result.get('run_id_source'))
    print('Training executed scopes:', result.get('training_executed_count'))
    print('Training reused scopes:', result.get('training_skipped_count'))
    print('All training scopes reused:', result.get('all_training_scopes_reused'))
    display(result['comparison'])
    if not result.get('missing_selected_profiles').empty:
        display(result['missing_selected_profiles'])
    if not result.get('seed_stability').empty:
        display(result['seed_stability'])
    if not result.get('seed_ensemble').empty:
        display(result['seed_ensemble'])
    if not result.get('experiment_policy_guard').empty:
        print('Experiment policy guard:')
        display(result['experiment_policy_guard'])
    if not result.get('future_oos_candidate_plan').empty:
        print('Future OOS candidate plan:')
        display(result['future_oos_candidate_plan'])
    if not result.get('performance_gap_analysis').empty:
        print('Performance gap analysis:')
        display(result['performance_gap_analysis'])
    if not result.get('payoff_alignment_summary').empty:
        print('Payoff alignment summary:')
        display(result['payoff_alignment_summary'])
    if not result.get('payoff_policy_robustness_summary').empty:
        print('Payoff policy robustness summary:')
        display(result['payoff_policy_robustness_summary'])
    print('Decision:', result['decision'])
finally:
    runtime.unassign()


In [ ]:
# Runtime is released once, after the experiment matrix cell finishes.
# Outputs are profile-isolated under: {CHECKPT_DIR}/experiments/{run_id}/{profile}/{fold_scope}/
